# Chapter 8: Transforming Columns

The programs from Chapter 8 of *Data Analysis with Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name.

Run the cells from top to bottom (**Run > Run All Cells**). The programs read the datasets from `../data/`, so keep this notebook in its chapter folder.

## New Columns From Old Ones

`new_columns.py`

In [1]:
import numpy as np
import pandas as pd

orders = pd.read_csv("../data/orders.csv", parse_dates=["order_date"])

orders["revenue"] = (
    orders["quantity"] * orders["unit_price"] * (1 - orders["discount"])
)
orders = orders.assign(
    month=orders["order_date"].dt.month,
    weekend=orders["order_date"].dt.dayofweek >= 5,
)
orders["size"] = np.select(
    [orders["revenue"] < 30, orders["revenue"] < 100],
    ["small", "medium"],
    default="large",
)
orders["channel"] = orders["channel"].map(
    {"web": "Website", "app": "Mobile app", "marketplace": "Marketplace"}
)
cols = ["order_id", "revenue", "month", "weekend", "size", "channel"]
print(orders[cols].head(4))
print(orders["size"].value_counts())

  order_id  revenue  month  weekend    size     channel
0   L00001     24.0      1    False   small  Mobile app
1   L00002     21.6      1    False   small     Website
2   L00003     32.0      1    False  medium     Website
3   L00004     18.0      1    False   small  Mobile app
size
medium    3260
small     1475
large     1098
Name: count, dtype: int64


## Bins: Turning Numbers Into Groups

`bins.py`

In [2]:
import pandas as pd

customers = pd.read_csv("../data/customers.csv")
customers["age"] = 2025 - customers["birth_year"]

customers["age_band"] = pd.cut(
    customers["age"],
    bins=[0, 29, 44, 59, 120],
    labels=["under 30", "30-44", "45-59", "60+"],
)
print(customers["age_band"].value_counts().sort_index())

# qcut makes groups of (almost) equal size instead of equal width
customers["age_quartile"] = pd.qcut(
    customers["age"], 4, labels=["Q1", "Q2", "Q3", "Q4"]
)
print(
    customers.groupby("age_quartile", observed=True)["age"].agg(
        ["min", "max", "count"]
    )
)

age_band
under 30    235
30-44       341
45-59       364
60+         260
Name: count, dtype: int64
              min  max  count
age_quartile                 
Q1             20   32    305
Q2             33   45    298
Q3             46   58    306
Q4             59   70    291


## Row by Row, or Whole Columns?

`apply_vs_vector.py`

In [3]:
import pandas as pd

orders = pd.read_csv("../data/orders.csv")


def shipping(row):
    """Free shipping over $100; otherwise $6, or $4 for app orders."""
    total = row["quantity"] * row["unit_price"]
    if total >= 100:
        return 0.0
    return 4.0 if row["channel"] == "app" else 6.0


slow = orders.apply(shipping, axis=1)  # calls Python once per row

total = orders["quantity"] * orders["unit_price"]
fast = (
    pd.Series(6.0, index=orders.index)
    .mask(orders["channel"] == "app", 4.0)
    .mask(total >= 100, 0.0)
)  # whole-column operations

print("same answer:", slow.equals(fast))
print("total shipping charged:", fast.sum())

same answer: True
total shipping charged: 24808.0
